# Exploración inicial de China y España

Versión fija **v0_2026-10-16**. Se examinan cobertura, huecos, flags, tasas y diferencias entre fuentes. No se descarga nada ni se modifican bronze/silver/gold.

**Criterio de fuentes:** NBS directo como último recurso. Un distribuidor internacional puede reutilizar estadísticas nacionales; la coincidencia entre fuentes no prueba independencia ni descarta sesgos. Los saltos marcados son observaciones para revisar, no pruebas de manipulación.

Ejecutar con el entorno `uv sync --group analysis --locked`; seleccionar su kernel Python en Jupyter o VS Code. El notebook ejecutado conserva tablas y gráficos; para usar los controles, volver a ejecutar las celdas con un kernel activo.

In [1]:
from pathlib import Path
import sys, json
import polars as pl
from IPython.display import display, Markdown, clear_output
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/eda.py").exists())
sys.path.insert(0, str(root / "src"))
from snapshot import verify
from eda import load, coverage, explorer, cross_sources, shocks, demand_residuals, build
VERSION = "v0_2026-10-16"
manifest = verify(VERSION, root)
df, frozen = load(VERSION, root)
print(f"{VERSION}: {df.height:,} observaciones; {len(manifest['bronze_files'])} originales verificados")


v0_2026-10-16: 88,798 observaciones; 15 originales verificados


## 1. Procedencia y versión

Los metadatos distinguen proveedor, URL, fecha de acceso, fecha de actualización y hash. Una versión congelada garantiza que repetimos los mismos cálculos con los mismos bytes; no certifica la veracidad de la medición económica.

In [2]:
display(pl.DataFrame([{
    "dataset": x["metadata"].get("id", x["metadata"].get("dataset_id")),
    "provider": x["metadata"].get("provider"),
    "accessed": x["metadata"].get("accessed_at_utc"),
    "updated": x["metadata"].get("provider_updated_at"),
    "sha256": x["sha256"]
} for x in manifest["bronze_files"]], infer_schema_length=None))
policy = json.loads((frozen / "datos/catalogo/source_policy.json").read_text())
print(policy["upstream_origin"])


A different distributor is not evidence of independent measurement


dataset,provider,accessed,updated,sha256
str,str,str,str,str
"""estat_nama_10_a10""","""eurostat""","""2026-10-16T08:14:51.621482+00:…","""2026-10-15T23:00:00+0200""","""b49b2b92b0953c1dac66ae81b784b8…"
"""estat_nama_10_a10_e""","""eurostat""","""2026-10-16T08:16:16.091299+00:…","""2026-10-14T23:00:00+0200""","""1d792e5d3421c7752a25f0f986207d…"
"""estat_nama_10_gdp""","""eurostat""","""2026-10-16T08:14:52.077405+00:…","""2026-10-15T23:00:00+0200""","""d025963e76fb27e2855c8ee928ad87…"
"""estat_nama_10_nfa_st""","""eurostat""","""2026-10-16T08:14:55.690168+00:…","""2026-10-15T23:00:00+0200""","""0dd942852ba78a2f1caa4f1328930e…"
"""estat_nama_10_nfa_st""","""eurostat""","""2026-10-16T08:15:44.457827+00:…","""2026-10-15T23:00:00+0200""","""214d81ab99f766e0a7a5723498cfb4…"
…,…,…,…,…
"""local_f6cd6648d813""","""user_supplied""",null,null,"""f6cd6648d813b044f7eee925aa9ba6…"
"""pwt110""","""pwt""","""2026-10-16T08:15:00.781279+00:…",null,"""7b337e94f39dfe304bc459e433f8ce…"
"""pwt110_capital""","""pwt""","""2026-10-16T08:15:00.596747+00:…",null,"""37c7449ff30a91e4597c2a06fa947c…"


## 2. Cobertura y calidad técnica

La cobertura efectiva se calcula por serie y país, no por las columnas anunciadas en un CSV. Se detectan huecos internos y se conservan ceros, valores negativos y flags. Estos últimos no se convierten automáticamente en errores.

In [3]:
cov = coverage(df)
keys = ["rgdpna", "rnna", "emp", "avh", "q_gdp", "q_c", "q_i", "q_g", "q_x", "q_m", "SI.POV.GINI"]
display(cov.filter(pl.col("source").is_in(["pwt110", "pwt110_na", "wdi_common"]) & pl.col("series").is_in(keys)))
assert df["value"].is_finite().all()
assert not df.select(pl.struct(["country", "year", "series", "source"]).is_duplicated().any()).item()
print("Claves únicas y valores numéricos finitos.")


Claves únicas y valores numéricos finitos.


country,source,series,unit,first,last,n,internal_missing,missing_years,zeros,negative,nonempty_flags
str,str,str,str,i64,i64,i64,i64,str,i64,i64,i64
"""CHN""","""pwt110""","""avh""","""see_pwt_legend""",1952,2023,72,0,"""""",0,0,0
"""CHN""","""pwt110""","""emp""","""see_pwt_legend""",1952,2023,72,0,"""""",0,0,0
"""CHN""","""pwt110""","""rgdpna""","""see_pwt_legend""",1952,2023,72,0,"""""",0,0,0
"""CHN""","""pwt110""","""rnna""","""see_pwt_legend""",1952,2023,72,0,"""""",0,0,0
"""CHN""","""pwt110_na""","""avh""","""see_pwt_legend""",1952,2025,74,0,"""""",0,0,0
…,…,…,…,…,…,…,…,…,…,…,…
"""ESP""","""pwt110_na""","""q_gdp""","""see_pwt_legend""",1950,2023,74,0,"""""",0,0,0
"""ESP""","""pwt110_na""","""q_i""","""see_pwt_legend""",1950,2023,74,0,"""""",0,0,0
"""ESP""","""pwt110_na""","""q_m""","""see_pwt_legend""",1950,2023,74,0,"""""",0,0,0


## 3. Exploración de series

Se empieza con índices 2008=100. Las tasas requieren años consecutivos y niveles positivos. Si falta la base elegida, se informa; no se sustituye silenciosamente. Los niveles se muestran en paneles separados por país para evitar superponer euros y yuanes.

El selector permite escoger proveedor, serie, transformación y período. PWT contiene variables a PPA y otras a precios nacionales; consultar la leyenda conservada antes de interpretar niveles.

In [4]:
explorer(df, "pwt110_na", "q_gdp", "index").show(renderer="plotly_mimetype")


In [5]:
import ipywidgets as widgets
sources = [s for s in ["pwt110_na", "pwt110", "wdi_common", "estat_nama_10_gdp"] if s in df["source"].unique()]
source = widgets.Dropdown(options=sources, value="pwt110_na", description="Proveedor")
series = widgets.Dropdown(description="Serie", layout=widgets.Layout(width="650px"))
mode = widgets.Dropdown(options=[("Índice", "index"), ("Crecimiento anual", "growth"), ("Nivel por país", "level")], value="index", description="Medida")
years = widgets.IntRangeSlider(value=[1990, 2023], min=1960, max=2025, description="Años", continuous_update=False)
base_year = widgets.IntSlider(value=2008, min=1960, max=2023, description="Base", continuous_update=False)
out = widgets.Output()
def render(*_):
    with out:
        clear_output(wait=True)
        if not series.value:
            return
        print("Unidad almacenada:", df.filter((pl.col("source") == source.value) & (pl.col("series") == series.value))["unit"].unique().to_list())
        try:
            explorer(df, source.value, series.value, mode.value, *years.value, base_year.value).show(renderer="plotly_mimetype")
        except ValueError as error:
            print(error)
def update_series(*_):
    options = sorted(df.filter(pl.col("source") == source.value)["series"].unique().to_list())
    series.options = options
    series.value = "q_gdp" if "q_gdp" in options else options[0]
    render()
source.observe(update_series, names="value")
for control in [series, mode, years, base_year]:
    control.observe(render, names="value")
update_series()
display(widgets.VBox([source, series, mode, years, base_year, out]))


Unidad almacenada: ['see_pwt_legend']
Unidad almacenada: ['see_pwt_legend']


## 4. Diferencias entre fuentes

Se comparan tasas de crecimiento del PIB real durante 1990–2023, no niveles de monedas distintas. La referencia WDI no se considera verdad observada. Diferencias entre ediciones y ajustes metodológicos pueden explicar divergencias; el acuerdo entre fuentes también puede proceder de inputs comunes.

In [6]:
cross = cross_sources(df)
summary = cross.group_by(["country", "source"]).agg(
    pl.len().alias("n"),
    pl.col("difference_pp").abs().mean().alias("mean_abs_difference_pp"),
    pl.col("difference_pp").abs().max().alias("max_abs_difference_pp")
).sort(["country", "source"])
display(summary)
display(cross.sort(pl.col("difference_pp").abs(), descending=True).head(12))


country,source,n,mean_abs_difference_pp,max_abs_difference_pp
str,str,u32,f64,f64
"""CHN""","""pwt110""",34,0.064476,0.183183
"""CHN""","""pwt110_na""",34,0.064476,0.183181
"""ESP""","""estat_nama_10_gdp""",28,0.003604,0.100903
"""ESP""","""pwt110""",34,0.01194,0.214802
"""ESP""","""pwt110_na""",34,0.011939,0.214803


country,year,source,series,wdi_growth,other_growth,difference_pp
str,i64,str,str,f64,f64,f64
"""ESP""",2023,"""pwt110_na""","""q_gdp""",2.460858,2.675661,0.214803
"""ESP""",2023,"""pwt110""","""rgdpna""",2.460858,2.675659,0.214802
"""ESP""",2022,"""pwt110""","""rgdpna""",6.370317,6.179319,-0.190998
"""ESP""",2022,"""pwt110_na""","""q_gdp""",6.370317,6.17932,-0.190997
"""CHN""",2022,"""pwt110""","""rgdpna""",3.133854,2.950672,-0.183183
…,…,…,…,…,…,…
"""CHN""",2023,"""pwt110""","""rgdpna""",5.416003,5.249565,-0.166438
"""CHN""",2021,"""pwt110_na""","""q_gdp""",8.570384,8.448471,-0.121913
"""CHN""",2021,"""pwt110""","""rgdpna""",8.570384,8.448473,-0.121911


## 5. Saltos e identidades

Se marcan crecimientos de magnitud superior al 15 % o desviaciones robustas superiores a seis, calculadas con mediana y MAD de la serie. Es un cribado descriptivo: las crisis y rebotes pueden ser auténticos. No se elimina ni corrige ninguna observación.

La identidad de demanda se comprueba a precios corrientes. No se exige aditividad a los volúmenes encadenados.

In [7]:
events = shocks(df)
display(events)
residuals = demand_residuals(frozen)
display(residuals.filter(pl.col("year").is_between(1990, 2023)).group_by("country").agg(
    pl.col("nominal_residual_pct_gdp").abs().max().alias("max_abs_residual_pct_gdp")
))


country,source,series,year,growth_pct,robust_score,meaning
str,str,str,i64,f64,f64,str
"""CHN""","""pwt110""","""avh""",2005,4.607071,12.819879,"""review_only_not_error"""
"""CHN""","""pwt110""","""avh""",2007,-3.805497,10.481442,"""review_only_not_error"""
"""CHN""","""pwt110""","""avh""",2010,5.145414,14.310992,"""review_only_not_error"""
"""CHN""","""pwt110""","""avh""",2020,-4.35347,11.999231,"""review_only_not_error"""
"""ESP""","""pwt110""","""avh""",2020,-6.88523,9.943468,"""review_only_not_error"""
…,…,…,…,…,…,…
"""CHN""","""pwt110_na""","""q_i""",2007,15.219226,0.726499,"""review_only_not_error"""
"""CHN""","""pwt110_na""","""q_i""",2009,19.081089,1.191789,"""review_only_not_error"""
"""CHN""","""pwt110_na""","""q_i""",2010,15.204932,0.724776,"""review_only_not_error"""


country,max_abs_residual_pct_gdp
str,f64
"""ESP""",0.240376
"""CHN""",1.189995


## 6. Resultados de la exploración

Los productos EDA se escriben en un directorio separado por versión. No sustituyen los archivos congelados. La numeración del informe se pospone: se conservará el identificador del profesor, con sufijos de país como `2.B.1-CHN` y `2.B.1-ESP`, o paneles dentro de una misma figura.

In [8]:
result = build(VERSION, root)
print(result)
display(Markdown((Path(result["output"]) / "HALLAZGOS.md").read_text()))


{'version': 'v0_2026-10-16', 'observations': 88798, 'series': 2600, 'events': 16, 'output': '/home/xylon/.codex/.chatgpt-projects/g-p-6aa31c45d53481919614ccfb0597081f/trabajo_china/output/eda/v0_2026-10-16'}


# Exploración preliminar de la versión v0_2026-10-16

## Alcance

- 88,798 observaciones almacenadas; se conservan variantes, países, conceptos y unidades por separado.
- Los gráficos principales emplean índices y tasas. Los niveles originales se separan por país.
- No se han descargado datos directos del NBS. Su utilización queda como último recurso.
- Esta exploración no demuestra ni refuta inflación estadística. Tampoco acredita independencia entre WDI y PWT.

## Diferencias de crecimiento del PIB frente a WDI

- CHN / pwt110: 34 años; diferencia absoluta media 0.0645 p.p.; máxima 0.1832 p.p.
- CHN / pwt110_na: 34 años; diferencia absoluta media 0.0645 p.p.; máxima 0.1832 p.p.
- ESP / estat_nama_10_gdp: 28 años; diferencia absoluta media 0.0036 p.p.; máxima 0.1009 p.p.
- ESP / pwt110: 34 años; diferencia absoluta media 0.0119 p.p.; máxima 0.2148 p.p.
- ESP / pwt110_na: 34 años; diferencia absoluta media 0.0119 p.p.; máxima 0.2148 p.p.

## Revisión prioritaria

- 16 eventos marcados con reglas de cribado explícitas. Revisar crisis, cambios de definición y revisiones; no corregir automáticamente.
- Las diferencias entre fuentes pueden reflejar distintas ediciones y métodos; no constituyen errores por sí mismas.
- Los residuos de la identidad nominal se calculan en puntos de PIB; no se impone aditividad a los volúmenes encadenados.
- Las variables con huecos no se interpolan. Los flags de Eurostat se conservan y se consultan en su respuesta original.
- Antes de cerrar el informe faltan: cadena de procedencia CHN, validación de universos laborales, existencias/CCF reales, rentas y tablas input-output.

## Archivos

- coverage_gaps_flags.csv: cobertura efectiva, huecos internos, ceros, negativos y flags por serie.
- gdp_source_differences.csv y gdp_source_summary.csv: comparación de tasas sin igualar niveles de moneda.
- review_events.csv: revisión de saltos; no identifica automáticamente errores.
- nominal_demand_residuals.csv: identidad de demanda corriente.
- exploracion.html: gráficos interactivos autocontenidos.
